# Phase 3 — LiteCrack region-only baseline

This notebook embeds code, configs, frozen splits and the Phase 2 baseline registry.
Use a Kaggle GPU; enable Internet if datasets need restoring. MobileNetV3-Small
uses width multiplier 0.5, weights=None and GroupNorm, with a separable decoder.
No ImageNet download, boundary head, adaptation or query-guided tuning occurs.

The runtime must already contain compatible torch/torchvision. Do not upgrade
Kaggle's working GPU stack blindly. CPU/CUDA exact recovery, spatial shape,
gradient and parameter-budget tests gate the real training run.


In [1]:
import os,sys,json,hashlib,io,base64,zipfile,subprocess
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
project=Path('/kaggle/working/road_crack_phase3_v1')
project.mkdir(exist_ok=True)
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='6468cf0300e4647fd28942d0df20d76668369a7ccd8c3f8f76a1460bac311cec'
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        path=(project/name).resolve(); assert path.is_relative_to(project.resolve())
        if path.exists(): assert path.read_bytes()==z.read(name), 'Existing source differs: preserve and use a new project directory'
    z.extractall(project)
os.chdir(project)
import torch,torchvision
assert torch.cuda.is_available(), 'Enable Kaggle GPU'
print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__,'torchvision:',torchvision.__version__)
manifest=project/'data/manifests/pilot_v1.json'
restore_report=project/'artifacts/dataset_restore.json'
subprocess.run([sys.executable,'scripts/restore_datasets.py','--manifest',str(manifest),
    '--sources','configs/data_sources.json','--destination','/kaggle/working/road_crack_phase1/data/raw',
    '--report',str(restore_report)],check=True)
data_root=Path(json.loads(restore_report.read_text())['data_root'])
evidence=project/'artifacts/phase3'
evidence.mkdir(parents=True,exist_ok=True)


GPU: Tesla T4 torch: 2.11.0+cu128 torchvision: 0.26.0+cu128
Reused existing data; all image/mask checksums passed: /kaggle/working/road_crack_phase1/data/raw


In [2]:
import xml.etree.ElementTree as ET
result=subprocess.run([sys.executable,'-m','pytest','-q','-rA','--junitxml='+str(evidence/'tests.xml')],
    text=True,capture_output=True)
(evidence/'tests.txt').write_text(result.stdout+result.stderr)
print(result.stdout,result.stderr)
assert result.returncode==0, 'Tests failed: do not start training'
cases=ET.parse(evidence/'tests.xml').findall('.//testcase')
recovery=[c for c in cases if 'test_actual_trainer_mid_epoch_resume' in c.attrib.get('name','')]
assert len(recovery)==6 and all(c.find('skipped') is None for c in recovery), 'All three model CPU/CUDA recovery tests must run'
compact=[c for c in cases if 'test_compact_' in c.attrib.get('name','')]
assert len(compact)==2 and all(c.find('skipped') is None for c in compact)
(evidence/'requirements.kaggle.lock.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
subprocess.run([sys.executable,'scripts/environment.py'],check=True)
sys.path.insert(0,str(project))
from src.model_factory import build_model
config=json.loads((project/'configs/litecrack.json').read_text())
model=build_model(config)
parameters=sum(p.numel() for p in model.parameters())
registry=json.loads((project/'data/manifests/baseline_registry_v1.json').read_text())
baseline_parameters=2110029
assert parameters<=config['max_parameters'] and parameters<=baseline_parameters/2
print('LiteCrack parameters:',parameters,'R2AU parameters:',baseline_parameters,'reduction:',1-parameters/baseline_parameters)
del model


........................                                                 [100%]
==================================== PASSES ====================================
________________ test_actual_trainer_mid_epoch_resume[cpu-unet] ________________
----------------------------- Captured stdout call -----------------------------
{"epoch": 1, "step": 2, "train_loss": 1.3289219737052917, "validation": {"loss": 0.8950339555740356, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "predicted_foreground_fraction": 0.0, "truth_foreground_fraction": 0.0625, "n": 2}, "learning_rate": 0.001}
{"epoch": 2, "step": 4, "train_loss": 1.300324559211731, "validation": {"loss": 0.8954219222068787, "dice": 0.0, "iou": 0.0, "precision": 0.0, "recall": 0.0, "predicted_foreground_fraction": 0.0, "truth_foreground_fraction": 0.0625, "n": 2}, "learning_rate": 0.001}
{"epoch": 1, "step": 2, "train_loss": 1.3289219737052917, "validation": {"loss": 0.8950339555740356, "dice": 0.0, "iou": 0.0, "precision": 0.0, 

## Train one fixed configuration

Same seed, split, resolution, region objective, optimizer, clipping and stopping
budget as the stabilized baselines. Validation selects the checkpoint. Target
query and source holdout are not evaluated in this development run. No claim of
superiority follows from a smaller parameter count or one seed.

Every epoch and 30 optimizer steps saves immutable recovery state. Rerunning
resumes the same configuration. Historical snapshots are never pruned. Kernel
termination can lose work since the latest saved checkpoint; download outputs.


In [3]:
import shutil,signal
assert shutil.disk_usage('/kaggle/working').free>2*1024**3, 'Need at least 2 GiB free'
run=evidence/'litecrack'
command=[sys.executable,'scripts/train_baseline.py','--config','configs/litecrack.json',
    '--manifest',str(manifest),'--data-root',str(data_root),'--run-dir',str(run)]
if (run/'checkpoints/latest.json').exists(): command.append('--resume')
def execute_logged(command,path):
    with open(path,'a',buffering=1) as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        print('Worker PID:',process.pid,flush=True)
        try:
            for line in process.stdout: print(line,end='',flush=True); log.write(line)
            exit_code=process.wait()
        except KeyboardInterrupt:
            process.send_signal(signal.SIGTERM); process.wait(timeout=120); raise
    if exit_code: raise RuntimeError('Worker failed: '+str(path))
execute_logged(command,evidence/'training.log')
print((run/'status.json').read_text())
if (run/'evaluation.json').exists(): print((run/'evaluation.json').read_text())


Worker PID: 398
{"epoch": 1, "step": 30, "train_loss": 1.8729237119356792, "validation": {"loss": 0.9471038341522217, "dice": 0.05450623952007542, "iou": 0.028398608874219158, "precision": 0.02840078768786043, "recall": 0.9968600253264109, "predicted_foreground_fraction": 0.9996597290039062, "truth_foreground_fraction": 0.028463999430338543, "n": 60}, "learning_rate": 0.0001}
{"epoch": 2, "step": 60, "train_loss": 1.647370986143748, "validation": {"loss": 0.9446452935536702, "dice": 0.13200492829006785, "iou": 0.07630248613810788, "precision": 0.08466169342961317, "recall": 0.3466163589929541, "predicted_foreground_fraction": 0.14388961791992189, "truth_foreground_fraction": 0.028463999430338543, "n": 60}, "learning_rate": 0.0001}
{"epoch": 3, "step": 90, "train_loss": 1.5468554298082988, "validation": {"loss": 0.9404340187708536, "dice": 0.23513361280978037, "iou": 0.1520598755601289, "precision": 0.17874172703013755, "recall": 0.3695571820872525, "predicted_foreground_fraction": 0.06

In [4]:
assert json.loads((run/'status.json').read_text())['state']=='training_complete', 'Export safe-stop state before resuming'
for device in ['cuda','cpu']:
    execute_logged([sys.executable,'scripts/profile_baseline.py','--run-dir',str(run),'--device',device],
                   evidence/('profile_'+device+'.log'))


Worker PID: 407
{
  "architecture": "litecrack",
  "device": "cuda",
  "hardware": "Tesla T4",
  "threads": 4,
  "batch_size": 1,
  "image_size": 256,
  "parameters": 339265,
  "p50_ms": 8.52136999992581,
  "p95_ms": 9.778347999599646,
  "warmups": 20,
  "timed_iterations": 200,
  "scope": "model pass only; no image decoding/transfer",
  "recovery_checkpoint_bytes": 4319065,
  "fp32_weight_bytes": 1357060,
  "cuda_peak_allocated_bytes": 15592960,
  "conv_linear_macs": 148741376,
  "mac_convention": "One MAC per multiply-accumulate; Conv2d/Linear only; excludes normalization, activation, pooling, resize and elementwise operations"
}
Worker PID: 416
{
  "architecture": "litecrack",
  "device": "cpu",
  "hardware": "x86_64",
  "threads": 4,
  "batch_size": 1,
  "image_size": 256,
  "parameters": 339265,
  "p50_ms": 40.57263549975687,
  "p95_ms": 45.874787999309774,
  "warmups": 20,
  "timed_iterations": 200,
  "scope": "model pass only; no image decoding/transfer",
  "recovery_checkpoint_

## Export before ending the session

Run even if an earlier cell failed. The ZIP includes code, configs, tests, metrics,
and latest/best snapshots. Other historical snapshots remain on Kaggle; no files
are deleted. Download through Kaggle Files and save a persistent notebook output.
The working directory alone is not an external backup.


In [5]:
backup=Path('/kaggle/working/phase3_results.zip')
selected={p for p in project.rglob('*') if p.is_file() and p.suffix!='.pt'
          and '__pycache__' not in p.parts and '.pytest_cache' not in p.parts}
folder=evidence/'litecrack/checkpoints'
for which in ['latest','best']:
    pointer=folder/(which+'.json')
    if pointer.exists():
        entry=json.loads(pointer.read_text()); path=(folder/entry['file']).resolve()
        assert path.parent==folder.resolve()
        assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256']
        selected.add(path)
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(selected): z.write(p,p.relative_to(project).as_posix())
digest=hashlib.sha256(backup.read_bytes()).hexdigest()
backup.with_suffix('.zip.sha256').write_text(digest+'  '+backup.name+'\n')
print('Download:',backup)
print('SHA256:',digest)


Download: /kaggle/working/phase3_results.zip
SHA256: d1d9c4fa7c42efb64e2f30f775fbbf8f5973f22676f591ba1537523d63119470
